# Extracción de datos CAMELS-CL

Flujo interactivo para cargar PP, T, Q y ETP desde CAMELS-CL v1.0.

| Paso | Acción |
|------|--------|
| 1 | Buscar cuenca por nombre o código |
| 2 | Seleccionar cuenca y revisar metadatos |
| 3 | Elegir período y variables |
| 4 | Verificar cobertura de datos |
| 5 | Cargar y previsualizar |


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..') / 'src'))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec
from IPython.display import display, HTML

%matplotlib inline

from loader_camels import (
    listar_cuencas, cargar_atributos,
    cargar_forzantes, cargar_caudal,
    descargar_dataset,
)

_COL_PP  = '#1f77b4'
_COL_T   = '#c0392b'
_COL_Q   = '#27ae60'
_COL_ETP = '#ff7f0e'

CACHE_DIR = Path('C:/tmp/camels_cache')
print('Loader listo. Cache:', CACHE_DIR)

## Paso 1 — Búsqueda de cuenca


In [ ]:
BUSCAR = 'turbio'   # <-- cambiar aquí para simular la búsqueda

cat = listar_cuencas(CACHE_DIR, verbose=False)

resultado = cat[cat['nombre'].str.lower().str.contains(BUSCAR.lower())]

print(f"Cuencas encontradas para '{BUSCAR}': {len(resultado)}")
display(
    resultado[['nombre', 'lat', 'lon', 'area_km2', 'elev_media_m', 'periodo_inicio', 'periodo_fin']]
    .style
    .format({'lat': '{:.4f}', 'lon': '{:.4f}', 'area_km2': '{:.0f}', 'elev_media_m': '{:.0f}'})
    .set_table_styles([{'selector': 'th', 'props': [('font-size', '11px')]},
                       {'selector': 'td', 'props': [('font-size', '11px')]}])
)

## Paso 2 — Selección y metadatos


In [ ]:
GAUGE_ID  = '4308001'   # Rio Turbio En Varillar

meta = cat.loc[GAUGE_ID]

print('=' * 50)
print(f'  CUENCA SELECCIONADA: {meta["nombre"]}')
print('=' * 50)
print(f'  Código DGA   : {GAUGE_ID}')
print(f'  Lat / Lon    : {meta["lat"]:.4f} / {meta["lon"]:.4f}')
print(f'  Área         : {float(meta["area_km2"]):.0f} km²')
print(f'  Elev. media  : {float(meta["elev_media_m"]):.0f} m s.n.m.')
print(f'  Registro     : {meta["periodo_inicio"]}  →  {meta["periodo_fin"]}')
print(f'  Días obs.    : {meta["n_obs"]}')
print('=' * 50)

## Paso 3 — Configuración del período y variables


In [ ]:
INICIO    = '1985-01-01'
FIN       = '2015-12-31'
VARIABLES = ['precip_cr2', 'tmean_cr2', 'etp_har']  # PP + T + ETP

# Validar que el período está dentro del registro disponible
p_ini = pd.Timestamp(meta['periodo_inicio'])
p_fin = pd.Timestamp(meta['periodo_fin'])
req_ini = pd.Timestamp(INICIO)
req_fin = pd.Timestamp(FIN)

aviso_ini = ' ⚠ ANTES del inicio del registro' if req_ini < p_ini else ' ✓'
aviso_fin = ' ⚠ DESPUÉS del fin del registro'  if req_fin > p_fin else ' ✓'

print(f'Período solicitado : {INICIO} → {FIN}{aviso_ini}{aviso_fin}')
print(f'Registro disponible: {meta["periodo_inicio"]} → {meta["periodo_fin"]}')
print(f'Variables          : {VARIABLES}')

## Paso 4 — Verificación de cobertura


In [ ]:
# Descarga los zips necesarios si no están en caché
descargar_dataset(CACHE_DIR, variables=VARIABLES + ['q_m3s'], verbose=True)

# Carga para analizar cobertura
fz = cargar_forzantes(CACHE_DIR, GAUGE_ID, INICIO, FIN, variables=VARIABLES, verbose=False)
q  = cargar_caudal(CACHE_DIR, GAUGE_ID, INICIO, FIN, verbose=False)

n_dias = (pd.Timestamp(FIN) - pd.Timestamp(INICIO)).days + 1

cobertura = {
    'PP   (precip_cr2)' : fz['pp'].notna().sum(),
    'T    (tmean_cr2)'  : fz['tmean'].notna().sum(),
    'ETP  (har)'        : fz['etp'].notna().sum(),
    'Q    (DGA m³/s)'   : q.notna().sum(),
}

print(f'Período: {INICIO} → {FIN}  ({n_dias} días)\n')
print(f'  {"Variable":<22} {"Días con dato":>13}  {"Cobertura":>9}  {"NaN":>6}')
print('  ' + '-' * 56)
for var, n in cobertura.items():
    pct = n / n_dias * 100
    nan = n_dias - n
    barra = '█' * int(pct / 5) + '░' * (20 - int(pct / 5))
    print(f'  {var:<22} {n:>13,}  {pct:>8.1f}%  {nan:>6,}')

## Paso 5 — Previsualización de datos


In [ ]:
plt.rcParams.update({
    'font.size': 10, 'axes.grid': True,
    'grid.color': '#e0e0e0', 'grid.linewidth': 0.5,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.dpi': 130,
})

fig = plt.figure(figsize=(14, 9))
gs  = GridSpec(4, 1, figure=fig, hspace=0.08)

ax_pp  = fig.add_subplot(gs[0])
ax_t   = fig.add_subplot(gs[1], sharex=ax_pp)
ax_etp = fig.add_subplot(gs[2], sharex=ax_pp)
ax_q   = fig.add_subplot(gs[3], sharex=ax_pp)

# Resampleo anual para visualizar tendencias en período largo
pp_a   = fz['pp'].resample('ME').sum()
t_a    = fz['tmean'].resample('ME').mean()
etp_a  = fz['etp'].resample('ME').sum()
q_a    = q.resample('ME').mean()

# Panel PP
ax_pp.bar(pp_a.index, pp_a.values, width=25, color=_COL_PP, alpha=0.7, linewidth=0)
ax_pp.set_ylabel('PP\n[mm/mes]', fontsize=9)
ax_pp.set_title(
    f"{meta['nombre']}  |  gauge {GAUGE_ID}  |  "
    f"área {float(meta['area_km2']):.0f} km²  |  "
    f"elev. media {float(meta['elev_media_m']):.0f} m s.n.m.",
    fontsize=10, loc='left', pad=6
)

# Panel T
ax_t.plot(t_a.index, t_a.values, color=_COL_T, lw=1.0)
ax_t.fill_between(t_a.index, t_a.values, alpha=0.15, color=_COL_T)
ax_t.set_ylabel('Tmean\n[°C]', fontsize=9)
ax_t.axhline(0, color='#999', lw=0.6, ls='--')

# Panel ETP
ax_etp.bar(etp_a.index, etp_a.values, width=25, color=_COL_ETP, alpha=0.7, linewidth=0)
ax_etp.set_ylabel('ETP\n[mm/mes]', fontsize=9)

# Panel Q
ax_q.plot(q_a.index, q_a.values, color=_COL_Q, lw=1.0)
ax_q.fill_between(q_a.index, q_a.values, alpha=0.15, color=_COL_Q)
ax_q.set_ylabel('Q\n[m³/s]', fontsize=9)

# Eje X: solo en el panel inferior
for ax in [ax_pp, ax_t, ax_etp]:
    plt.setp(ax.get_xticklabels(), visible=False)
ax_q.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax_q.xaxis.set_major_locator(mdates.YearLocator(5))

fig.text(0.01, 0.5, 'Fuente: CAMELS-CL v1.0 (Alvarez-Garreton et al., 2018)',
         ha='left', va='center', rotation='vertical', fontsize=8, color='#888')

plt.tight_layout()
plt.show()